# Exp 3 — Experimental Data Pipeline

**Source:** `exp_data/00_raw.xlsx`

Each checkpoint is saved as a numbered xlsx file so you can open any step in Excel and go back if needed.

| Step | File | Description |
|------|------|-------------|
| 00 | `00_raw.xlsx` | Experimental rows from all_data_raw (no question rows) |
| 01 | `01_filtered.xlsx` | One row per trial, selected columns, acc + rt added |
| 02 | `02_excl_acc.xlsx` | Exclude subjects below 85% accuracy threshold |
| 03 | `03_excl_likert.xlsx` | Exclude careless Likert responders (aligned with questions pipeline) |
| 04 | `04_excl_errors.xlsx` | Remove error trials (acc = 0) |
| 05 | `05_gaze_congruency.xlsx` | Add gaze_congruency, action_consistency, and initiation_time |
| 06 | `06_trimmed.xlsx` | RT outlier trimming — **Python ±2SD**, per subject × condition (SPSS's `2w.SPS` script has a confirmed bug, see `SPSS_2w_script_bug_summary.txt` in the project root — it under-trims by silently keeping RT=0 trials that its own filter logic flags for exclusion) |
| 07 | `07_no_outliers.xlsx` | Participant-level outlier removal (mean RT > grand mean + 2 SD) |
| 08 | `08_ranova_input.xlsx` | Mean RT per subject × condition — long + wide, ready for rANOVA |
| 09 | `09_initiation_time.xlsx` | Mean initiation time per subject, action trials only (ANCOVA covariate) |
| 10 | `10_questions_data_added.xlsx` | Wide RT data + question ratings + initiation-time covariate merged |
| 11 | `11_merged_initTime_for_ANCOVA.xlsx` | Stable xlsx copy of the merged data (same content as Step 10) — kept in `exp_data/`, won't change if the pipeline re-runs |
| — | `jasp/analysis.csv` | Same data as CSV — open in JASP to analyse (canonical, overwritten every run) |

In [1]:
import os
import pandas as pd
import numpy as np

base = r'C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment3\data_processing\exp_data'

## Step 01 — Filter to one row per trial, select columns, add acc/rt

**Filters applied:**
1. `Procedure[SubTrial]` = `'pro'` → keeps only experimental sub-trial rows
2. `Running[Trial]` = `'Exp'` → excludes practice trials (PracticeL)
3. `pic` = `'6L'` or `'6R'` → keeps only the last slide per trial (where the response was captured)

**acc/rt logic:** `Running[SubTrial]` says which side was last shown (ListLeft or ListRight). The corresponding `ColorSlideLeft` or `ColorSlideRight` columns hold the actual response data.

In [2]:
df = pd.read_excel(f'{base}\\00_raw.xlsx')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} cols')

Loaded: 102424 rows × 79 cols


In [3]:
# Filter 1: keep only experimental sub-trial rows
df = df[df['Procedure[SubTrial]'] == 'pro'].copy()
print(f'After Procedure[SubTrial]=pro: {len(df)} rows')

# Filter 2: exclude practice trials
df = df[df['Running[Trial]'] == 'Exp'].copy()
print(f'After Running[Trial]=Exp:      {len(df)} rows')

# Filter 3: keep only last-slide rows (where response was captured)
df = df[df['pic'].isin(['6L', '6R'])].copy()
print(f'After pic in [6L, 6R]:         {len(df)} rows')
print(f'Expected:                       {60 * 168} rows (60 subjects × 168 trials)')

After Procedure[SubTrial]=pro: 102424 rows
After Running[Trial]=Exp:      72912 rows
After pic in [6L, 6R]:         10416 rows
Expected:                       10080 rows (60 subjects × 168 trials)


In [5]:
# Verify: each subject should have exactly 168 rows
trial_counts = df.groupby('Subject').size()
bad = trial_counts[trial_counts != 168]
if bad.empty:
    print('All subjects have exactly 168 trials.')
else:
    print('WARNING — unexpected trial counts:')
    print(bad)

All subjects have exactly 168 trials.


In [6]:
# Select columns (Procedure[SubTrial] dropped after filtering)
keep = [
    'Subject', 'Handedness', 'blue', 'green',
    'Choice[Trial]', 'colorLeft', 'colorRight',
    'ColorSlideLeft.ACC', 'ColorSlideLeft.RESP', 'ColorSlideLeft.RT',
    'ColorSlideRight.ACC', 'ColorSlideRight.RESP', 'ColorSlideRight.RT',
    'corR', 'Exp.Sample', 'GazeColorConsistency',
    'Procedure[Trial]', 'SlideXeroA.RT', 'XeroDur',
    'pic', 'Running[SubTrial]'
]
df = df[keep].copy()
print(f'Columns selected: {df.shape[1]}')
df.head(3)

Columns selected: 21


,Subject,Handedness,blue,green,Choice[Trial],colorLeft,colorRight,ColorSlideLeft.ACC,ColorSlideLeft.RESP,ColorSlideLeft.RT,...,ColorSlideRight.RESP,ColorSlideRight.RT,corR,Exp.Sample,GazeColorConsistency,Procedure[Trial],SlideXeroA.RT,XeroDur,pic,Running[SubTrial]
146,1,right,a,d,Right,blue,white,0.0,NaN,0.0,...,a,397.0,a,1.0,offline,actionC,853.0,NaN,6R,ListRight
153,1,right,a,d,NaN,white,green,NaN,NaN,NaN,...,d,384.0,d,2.0,consinstent,controlRight,NaN,412.0,6R,ListRight
160,1,right,a,d,NaN,blue,white,1.0,a,348.0,...,NaN,NaN,a,3.0,consinstent,controlLeft,NaN,500.0,6L,ListLeft


In [7]:
# Calculate merged acc and rt based on which side recorded the response
left  = df['Running[SubTrial]'] == 'ListLeft'
right = df['Running[SubTrial]'] == 'ListRight'

df['acc'] = np.where(left, df['ColorSlideLeft.ACC'],
            np.where(right, df['ColorSlideRight.ACC'], np.nan))

df['rt']  = np.where(left, df['ColorSlideLeft.RT'],
            np.where(right, df['ColorSlideRight.RT'], np.nan))

print(f'acc nulls: {df["acc"].isna().sum()}  |  rt nulls: {df["rt"].isna().sum()}')
print(f'acc values: {df["acc"].value_counts().to_dict()}')

acc nulls: 0  |  rt nulls: 0
acc values: {1.0: 9513, 0.0: 903}


In [8]:
# ACC per subject
acc_per_sub = df.groupby('Subject')['acc'].mean().mul(100).round(2).reset_index()
acc_per_sub.columns = ['Subject', 'ACC_%']
acc_per_sub['flag'] = acc_per_sub['ACC_%'].apply(
    lambda x: '*** <80%' if x < 80 else ('* <85%' if x < 85 else '')
)
print(acc_per_sub.to_string(index=False))

 Subject  ACC_%     flag
       1  95.24         
       2  97.02         
       3  96.43         
       4  99.40         
       5  89.88         
       6  95.24         
       7  89.88         
       8  97.02         
       9  90.48         
      10  77.38 *** <80%
      11  87.50         
      12  92.26         
      13  97.62         
      14  85.12         
      15  80.95   * <85%
      16  97.62         
      17  94.05         
      18  95.24         
      19  93.45         
      20  91.07         
      21  83.33   * <85%
      22  92.86         
      23  95.24         
      24  83.93   * <85%
      25  97.62         
      26  78.57 *** <80%
      27  98.81         
      28  98.21         
      29  97.02         
      30  98.21         
      31  45.24 *** <80%
      32  91.67         
      33  92.26         
      40  80.95   * <85%
      41  92.26         
      42  99.40         
      43  94.05         
      44  99.40         
      45  86.31         


In [9]:
# Save checkpoint 01
out = f'{base}\\01_filtered.xlsx'
df.to_excel(out, index=False)
print(f'Saved: {out}')
print(f'Shape: {df.shape}')

Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment3\data_processing\exp_data\01_filtered.xlsx
Shape: (10416, 23)


## Step 02 — Exclude participants below 85% accuracy

Threshold: **85%** (same as Experiment 1).

Questions pipeline doesn't need to know about this exclusion — low-accuracy subjects naturally
drop out later when Step 10's merge keeps only subjects present in this (exp_data's) final sample.

In [10]:
df = pd.read_excel(f'{base}\\01_filtered.xlsx')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} cols')

# ACC per subject
acc_per_sub = df.groupby('Subject')['acc'].mean().mul(100).round(2)

# Identify subjects below threshold
THRESHOLD = 85.0
exclude = acc_per_sub[acc_per_sub < THRESHOLD]
keep_subs = acc_per_sub[acc_per_sub >= THRESHOLD].index

print(f'\nThreshold: {THRESHOLD}%')
print(f'\nExcluded subjects ({len(exclude)}):')
for sub, acc in exclude.sort_values().items():
    print(f'  ss {sub}: {acc:.2f}%')

print(f'\nRemaining: {len(keep_subs)} subjects')

Loaded: 10416 rows × 23 cols

Threshold: 85.0%

Excluded subjects (8):
  ss 68: 5.36%
  ss 31: 45.24%
  ss 10: 77.38%
  ss 26: 78.57%
  ss 15: 80.95%
  ss 40: 80.95%
  ss 21: 83.33%
  ss 24: 83.93%

Remaining: 54 subjects


In [11]:
df2 = df[df['Subject'].isin(keep_subs)].copy()

n_trials = len(df2)
n_subjects = df2['Subject'].nunique()
print(f'Remaining after exclusion:')
print(f'  Subjects: {n_subjects}')
print(f'  Trials:   {n_trials}  ({n_subjects} × 168 = {n_subjects * 168})')

out = f'{base}\\02_excl_acc.xlsx'
df2.to_excel(out, index=False)
print(f'\nSaved: {out}')

Remaining after exclusion:
  Subjects: 54
  Trials:   9072  (54 × 168 = 9072)

Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment3\data_processing\exp_data\02_excl_acc.xlsx


## Step 03 — Exclude careless Likert responders

Reads the excluded-subject list `questions/pipeline.ipynb` Step 03 writes directly
(`03_likert_excluded_subjects.txt`) — no recomputation from the report, no hardcoded list.

**Excluded (3):** ss 14, 53, 69
**Kept despite flag:** subjects with dominant_% > 75 but range > 0.5 — still discriminate between conditions

In [12]:
with open(os.path.join(os.path.dirname(base), 'questions', '03_likert_excluded_subjects.txt')) as f:
    content = f.read().strip()
    LIKERT_EXCLUDED = [int(x) for x in content.split(',') if x] if content else []
print(f'Read from questions pipeline: {LIKERT_EXCLUDED}')

df = pd.read_excel(f'{base}\\02_excl_acc.xlsx')
df3 = df[~df['Subject'].isin(LIKERT_EXCLUDED)].copy()

n_subjects = df3['Subject'].nunique()
n_trials = len(df3)
print(f'Remaining subjects: {n_subjects}')
print(f'Remaining trials:   {n_trials}  ({n_subjects} x 168 = {n_subjects * 168})')

out = f'{base}\\03_excl_likert.xlsx'
df3.to_excel(out, index=False)
print(f'Saved: {out}')
print(f'Shape: {df3.shape}')

Read from questions pipeline: [14, 53, 69]
Remaining subjects: 51
Remaining trials:   8568  (51 x 168 = 8568)
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment3\data_processing\exp_data\03_excl_likert.xlsx
Shape: (8568, 23)


## Step 04 — Remove error trials

Error trials: `acc = 0` — participant pressed the wrong key or timed out.

Removed before RT analysis. The 85% threshold in Step 02 ensures all remaining subjects are high-performers overall; this step removes the residual incorrect trials within those subjects.

In [13]:
df = pd.read_excel(f'{base}\\03_excl_likert.xlsx')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} cols')

n_before = len(df)
n_errors = (df['acc'] == 0).sum()
print(f'\nError trials (acc = 0): {n_errors}  ({n_errors / n_before * 100:.2f}%)')

# Errors per subject (only subjects with at least one error)
errors_per_sub = df[df['acc'] == 0].groupby('Subject').size()
print(f'\nErrors per subject (non-zero only):')
print(errors_per_sub.to_string())

df4 = df[df['acc'] == 1].copy()

n_subjects = df4['Subject'].nunique()
n_trials = len(df4)
print(f'\nRemaining after error removal:')
print(f'  Subjects: {n_subjects}')
print(f'  Trials:   {n_trials}  (removed {n_before - n_trials} error trials)')

out = f'{base}\\04_excl_errors.xlsx'
df4.to_excel(out, index=False)
print(f'\nSaved: {out}')
print(f'Shape: {df4.shape}')

Loaded: 8568 rows × 23 cols

Error trials (acc = 0): 428  (5.00%)

Errors per subject (non-zero only):
Subject
1      8
2      5
3      6
4      1
5     17
6      8
7     17
8      5
9     16
11    21
12    13
13     4
16     4
17    10
18     8
19    11
20    15
22    12
23     8
25     4
27     2
28     3
29     5
30     3
32    14
33    13
41    13
42     1
43    10
44     1
45    23
46     2
47    18
48    16
49     7
50     4
52     6
54     2
55    10
56     4
57    22
58     6
59     2
60     2
61    22
63     7
64     8
65     1
66     3
67     5

Remaining after error removal:
  Subjects: 51
  Trials:   8140  (removed 428 error trials)

Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment3\data_processing\exp_data\04_excl_errors.xlsx
Shape: (8140, 23)


## Step 05 — Calculate gaze_congruency and action_consistency

Logic follows Experiment 1 (`raw+processed.xlsx`, tab `cleaned`):

| Column | Source | Logic |
|--------|--------|-------|
| `actual_color_location` | `colorLeft` | `colorLeft == "white"` → "right", else "left" |
| `actual_gaze_dir` | `Running[SubTrial]` | `ListLeft` → "left", `ListRight` → "right" |
| `wanted_gaze_dir` | `Choice[Trial]` / `Procedure[Trial]` | action: `Choice[Trial].lower()`; control: from condition name |
| `gaze_congruency` | derived | `actual_color_location == actual_gaze_dir` → "congruent", else "incongruent" |
| `action_consistency` | `Procedure[Trial]` | actionC → "consistent", actionNC → "inconsistent", control → "control" |

`gaze_congruency` is calculated from raw columns — **not** from E-Prime's `GazeColorConsistency` (which has an "offline" category and a typo). `action_consistency` is a direct mapping — simpler than Exp 1's formula.

In [14]:
df = pd.read_excel(f'{base}\\04_excl_errors.xlsx')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} cols')

# actual_color_location: which side did the target color appear on?
# colorLeft = "white" means no color on left → color is on the right
df['actual_color_location'] = np.where(df['colorLeft'] == 'white', 'right', 'left')

# actual_gaze_dir: which direction did the gaze end up?
df['actual_gaze_dir'] = np.where(df['Running[SubTrial]'] == 'ListLeft', 'left', 'right')

# wanted_gaze_dir: intended direction (action: from Choice[Trial]; control: from condition name)
df['wanted_gaze_dir'] = np.where(
    df['Choice[Trial]'].isin(['Left', 'Right']),
    df['Choice[Trial]'].str.lower(),
    np.where(df['Procedure[Trial]'] == 'controlLeft',  'left',
    np.where(df['Procedure[Trial]'] == 'controlRight', 'right', ''))
)

# gaze_congruency
df['gaze_congruency'] = np.where(
    df['actual_color_location'] == df['actual_gaze_dir'], 'congruent', 'incongruent'
)

# action_consistency: direct mapping from Procedure[Trial]
df['action_consistency'] = df['Procedure[Trial]'].map({
    'actionC':      'consistent',
    'actionNC':     'inconsistent',
    'controlLeft':  'control',
    'controlRight': 'control',
})

# initiation_time: merge SlideXeroA.RT (action trials) and XeroDur (control trials)
# SlideXeroA.RT is NaN for control trials; XeroDur is NaN for action trials
# fillna picks the correct value for each condition automatically
df['initiation_time'] = df['SlideXeroA.RT'].fillna(df['XeroDur'])

# Sanity checks
print('\ngaze_congruency:')
print(df['gaze_congruency'].value_counts())
print('\naction_consistency:')
print(df['action_consistency'].value_counts())

# Cross-check: compare calculated gaze_congruency vs E-Prime GazeColorConsistency
print('\nE-Prime GazeColorConsistency vs calculated gaze_congruency:')
print(pd.crosstab(df['GazeColorConsistency'], df['gaze_congruency'], margins=True))

# initiation_time check: should be non-null for all rows
print(f'\ninitiation_time — missing: {df["initiation_time"].isna().sum()}')
print(df.groupby('action_consistency')['initiation_time'].describe()[['count', 'mean', 'min', 'max']].round(1))

out = f'{base}\\05_gaze_congruency.xlsx'
df.to_excel(out, index=False)
print(f'\nSaved: {out}')
print(f'Shape: {df.shape}')

Loaded: 8140 rows × 23 cols

gaze_congruency:
gaze_congruency
incongruent    4073
congruent      4067
Name: count, dtype: int64

action_consistency:
action_consistency
inconsistent    2745
consistent      2727
control         2668
Name: count, dtype: int64

E-Prime GazeColorConsistency vs calculated gaze_congruency:
gaze_congruency       congruent  incongruent   All
GazeColorConsistency                              
consinstent                1335            0  1335
inconsistent                  0         1333  1333
offline                    2732         2740  5472
All                        4067         4073  8140

initiation_time — missing: 0
                     count   mean    min     max
action_consistency                              
consistent          2727.0  644.9    2.0  2854.0
control             2668.0  451.0  350.0   550.0
inconsistent        2745.0  654.0    4.0  2908.0

Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment3\data

## Step 06 — RT outlier trimming (Python ±2SD)

For each `Subject × gaze_congruency × action_consistency` cell, drop trials more than 2 SD from
that cell's own mean, then take the mean of what remains.

**This replaces the manual SPSS step.** SPSS's `2w.SPS` script has a confirmed bug (see
`SPSS_2w_script_bug_summary.txt`): its last line, `compute FiltRT = filt*rt.`, relies on
multiplication to propagate the exclusion flag — but SPSS treats `SYSMIS * 0` as `0`, not `SYSMIS`.
Any trial with a genuine RT of exactly 0 (a missing-response code, not a real reaction time) that
SPSS's own filter correctly flags for exclusion ends up silently kept as a literal zero instead.
Python trimming is now canonical for all three experiments — no SPSS round-trip, no numeric w1/w2
recoding needed (the previous "export to SPSS format" step is gone; grouping is done directly on
the string labels `gaze_congruency`/`action_consistency`).

In [15]:
df = pd.read_excel(f'{base}\\05_gaze_congruency.xlsx')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} cols')

grp = df.groupby(['Subject', 'gaze_congruency', 'action_consistency'])['rt']
stats = grp.agg(['mean', 'std']).reset_index()
df = df.merge(stats, on=['Subject', 'gaze_congruency', 'action_consistency'])
keep_mask = (df['rt'] >= df['mean'] - 2*df['std']) & (df['rt'] <= df['mean'] + 2*df['std'])
n_before = len(df)
trimmed = df[keep_mask].copy()
n_after = len(trimmed)
print(f'Trials before trim: {n_before}')
print(f'Trials after +/-2SD trim: {n_after}  (removed {n_before-n_after}, {(n_before-n_after)/n_before*100:.2f}%)')

cellmeans = trimmed.groupby(['Subject', 'gaze_congruency', 'action_consistency'])['rt'].mean().reset_index()
cellmeans.columns = ['sub', 'gaze_congruency', 'action_consistency', 'rt_mean']
print(f'Cell means shape: {cellmeans.shape}')

out = f'{base}\\06_trimmed.xlsx'
cellmeans.to_excel(out, index=False)
print(f'Saved: {out}')

Loaded: 8140 rows × 29 cols
Trials before trim: 8140
Trials after +/-2SD trim: 7743  (removed 397, 4.88%)
Cell means shape: (306, 4)
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment3\data_processing\exp_data\06_trimmed.xlsx


## Step 07 — Participant-level mean RT outlier removal

Excludes participants whose overall mean (mean of their 6 cell means) exceeds grand mean + 2 SD.

Note: with Python trimming now canonical, only **2** subjects are excluded here (ss11, ss32) instead
of the earlier SPSS-based run's 3 (ss11, ss32, ss48) — final N moves from 48 to **49**. ss48's mean
RT no longer crosses the cutoff once the 37 phantom-zero trials are correctly removed from the
pool feeding into these means.

In [16]:
df = pd.read_excel(f'{base}\\06_trimmed.xlsx')
subj_means = df.groupby('sub')['rt_mean'].mean().reset_index()
subj_means.columns = ['sub', 'overall_mean_rt']
group_mean = subj_means['overall_mean_rt'].mean()
group_sd = subj_means['overall_mean_rt'].std()
cutoff = group_mean + 2*group_sd
excluded = subj_means[subj_means['overall_mean_rt'] > cutoff]
exclude_ids = excluded['sub'].tolist()
print(f'Grand mean RT : {group_mean:.1f} ms')
print(f'Grand SD      : {group_sd:.1f} ms')
print(f'Cutoff        : {cutoff:.1f} ms (mean + 2 SD)')
print(f'Excluded ({len(excluded)} subject(s)):')
for _, row in excluded.iterrows():
    print(f'  ss {int(row["sub"])}: mean RT = {row["overall_mean_rt"]:.1f} ms (+{row["overall_mean_rt"]-cutoff:.1f} ms over cutoff)')
clean = df[~df['sub'].isin(exclude_ids)].copy()
print(f'Remaining: {clean["sub"].nunique()} subjects, {len(clean)} rows')
out = f'{base}\\07_no_outliers.xlsx'
clean.to_excel(out, index=False)
print(f'Saved: {out}')

Grand mean RT : 529.5 ms
Grand SD      : 96.9 ms
Cutoff        : 723.2 ms (mean + 2 SD)
Excluded (2 subject(s)):
  ss 11: mean RT = 790.4 ms (+67.2 ms over cutoff)
  ss 32: mean RT = 748.1 ms (+24.9 ms over cutoff)
Remaining: 49 subjects, 294 rows
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment3\data_processing\exp_data\07_no_outliers.xlsx


## Step 08 — Reshape to long + wide

Data is already one row per subject × condition after Step 06/07, and already string-labeled
(no numeric w1/w2 to decode anymore) — this step is now just a pivot to wide format.

In [17]:
df = pd.read_excel(f'{base}\\07_no_outliers.xlsx')
long = df[['sub', 'gaze_congruency', 'action_consistency', 'rt_mean']].copy()
n_subs = long['sub'].nunique()
print(f'Long format: {len(long)} rows ({n_subs} subjects x 6 conditions = {n_subs*6})')

col_map = {
    ('incongruent', 'inconsistent'): 'incong_incons', ('incongruent', 'consistent'): 'incong_cons', ('incongruent', 'control'): 'incong_ctrl',
    ('congruent', 'inconsistent'): 'cong_incons', ('congruent', 'consistent'): 'cong_cons', ('congruent', 'control'): 'cong_ctrl',
}
wide = df.pivot(index='sub', columns=['gaze_congruency', 'action_consistency'], values='rt_mean')
wide.columns = [col_map[c] for c in wide.columns]
wide = wide.reset_index()
print(f'Wide format: {wide.shape[0]} subjects x {wide.shape[1]} columns')

out = f'{base}\\08_ranova_input.xlsx'
with pd.ExcelWriter(out, engine='openpyxl') as writer:
    long.to_excel(writer, sheet_name='long', index=False)
    wide.to_excel(writer, sheet_name='wide', index=False)
print(f'Saved: {out}  (sheets: long, wide)')

Long format: 294 rows (49 subjects x 6 conditions = 294)
Wide format: 49 subjects x 7 columns
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment3\data_processing\exp_data\08_ranova_input.xlsx  (sheets: long, wide)


## Step 09 — Mean initiation time per subject (ANCOVA covariate)

For the planned ANCOVA, the covariate must be a single, comparable value per subject — and it
should only be computed from trials where it means what the theory says it means.
`initiation_time` is `SlideXeroA.RT` on action trials (a real behavioral measure) but `XeroDur` on
control trials (an experimenter-imposed random delay, unrelated to anything the participant did).
Averaging across both would mix a genuine measure with arbitrary noise.

So this computes the mean **only across action trials** (consistent + inconsistent) — control
trials are excluded from this calculation entirely, and any ANCOVA using this covariate should
itself be restricted to the action-condition cells.

In [18]:
df = pd.read_excel(f'{base}\\05_gaze_congruency.xlsx')
action_only = df[df['action_consistency'].isin(['consistent', 'inconsistent'])]
init_time = action_only.groupby('Subject')['initiation_time'].mean().reset_index()
init_time.columns = ['sub', 'mean_initiation_time']
print(f'Computed for {len(init_time)} subjects (from {len(action_only)} action trials)')
print(init_time.describe().to_string())
out = f'{base}\\09_initiation_time.xlsx'
init_time.to_excel(out, index=False)
print(f'Saved: {out}')

Computed for 51 subjects (from 5472 action trials)
             sub  mean_initiation_time
count  51.000000             51.000000
mean   34.784314            652.785630
std    21.263409            258.847399
min     1.000000            286.485714
25%    16.500000            470.311897
50%    33.000000            585.008929
75%    54.500000            788.625199
max    67.000000           1567.619048
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment3\data_processing\exp_data\09_initiation_time.xlsx


## Step 10 — Merge question ratings + initiation time into wide RT data

Left join on `sub` — keeps only the 49 subjects present in the final exp data.

In [19]:
rt_wide = pd.read_excel(f'{base}\\08_ranova_input.xlsx', sheet_name='wide')
q_wide = pd.read_excel(os.path.join(os.path.dirname(base), 'questions', '04_aggregated.xlsx'), sheet_name='wide')
init_time = pd.read_excel(f'{base}\\09_initiation_time.xlsx')

merged = rt_wide.merge(q_wide, on='sub', how='left').merge(init_time, on='sub', how='left')
print(f'Subjects in exp data:       {rt_wide["sub"].nunique()}')
print(f'Subjects in questions data: {q_wide["sub"].nunique()}')
print(f'Subjects in merged output:  {merged["sub"].nunique()}')
dropped = set(q_wide['sub']) - set(rt_wide['sub'])
if dropped:
    print(f'Dropped from questions (not in exp data): ss {sorted(dropped)}')
missing_init = merged[merged['mean_initiation_time'].isna()]
if len(missing_init):
    print(f'WARNING: missing initiation time for: {missing_init["sub"].tolist()}')
print(f'Columns ({merged.shape[1]}): {list(merged.columns)}')

out = f'{base}\\10_questions_data_added.xlsx'
merged.to_excel(out, index=False)
print(f'Saved: {out}')

Subjects in exp data:       49
Subjects in questions data: 59
Subjects in merged output:  49
Dropped from questions (not in exp data): ss [10, 11, 15, 21, 24, 26, 31, 32, 40, 68]
Columns (20): ['sub', 'cong_cons', 'cong_ctrl', 'cong_incons', 'incong_cons', 'incong_ctrl', 'incong_incons', 'consistent_caused', 'consistent_control', 'consistent_predict', 'control_caused', 'control_control', 'control_predict', 'inconsistent_caused', 'inconsistent_control', 'inconsistent_predict', 'Age', 'Sex', 'Handedness', 'mean_initiation_time']
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment3\data_processing\exp_data\10_questions_data_added.xlsx


## Step 11 — Export to JASP

After opening in JASP: set `sub`, `Sex`, `Handedness` to Nominal. For the RT rANOVA, use the 6
`*_cons`/`*_incons`/`*_ctrl` columns. For the ANCOVA, use only the 4 action-condition columns
(`cong_cons`, `cong_incons`, `incong_cons`, `incong_incons`) with `mean_initiation_time` as
covariate — do not include the `*_ctrl` columns in that analysis (see Step 09 note).

**This re-exports the analysis file — any existing `.jasp` file built on the old SPSS-trimmed data
is now stale** and needs its analyses re-run against this new `analysis.csv`.

Also saves `11_merged_initTime_for_ANCOVA.xlsx` in `exp_data/` — a stable xlsx copy of the same merged data (same content as `10_questions_data_added.xlsx`), following the numbered-checkpoint naming convention, meant for continued JASP or Python analysis work without changing if the pipeline is re-run.

In [ ]:
jasp_dir = os.path.join(base, 'jasp')
os.makedirs(jasp_dir, exist_ok=True)
csv_path = os.path.join(jasp_dir, 'analysis.csv')
merged.to_csv(csv_path, index=False, encoding='utf-8')
print(f'Saved: {csv_path}')

working_path = f'{base}\\11_merged_initTime_for_ANCOVA.xlsx'
merged.to_excel(working_path, index=False)
print(f'Saved: {working_path}  (stable xlsx copy for continued JASP/Python analysis work)')
print(f'Subjects: {merged.shape[0]}  |  Columns: {merged.shape[1]}')